# StrOutputParser
> LLM이 생성한 응답을 문자열(String)로 변환하는 가장 단순한 출력 파서(output parser)입니다.

## Prompt

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

# chat프롬프트 템플릿
chat_prompt = ChatPromptTemplate.from_messages(
    [   
        # 모델에 대한 역할을 지정해줌
        SystemMessage(
            content=(
                "당신은 친근하고 도움이 되는 AI 어시스턴스입니다. 항상 한국어로 답변해주세요."
            )
        ),
        # 사람이 사용할 템플릿의 변수
        HumanMessagePromptTemplate.from_template("""
        {user_input}
        """),
    ]
)

In [3]:
# chat_prompt에 입력해야하는 변수
chat_prompt.input_variables

['user_input']

## Model

In [4]:
from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(
    model="gemma4:e4b",     # 모델 명
    temperature = 0.1,      # 값이 낮을 수록 정확하고 안정적인 답변이 나옴
    top_p=1.0,              # 전체 분포에서 정답을 출력함
    num_predict=1000,       # 크레딧을 1000에 제함함
    keep_alive="5m"         # 5분동안 모델을 계속 살려둠 다시 불러들일 때 더 빠름
)

## Chain without Parser
> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 결과 반환

In [ ]:
# chain 형성
# 형성된 프롬프트 | 형성된 모델
chain = chat_prompt | model

In [7]:
response = chain.invoke({
    "user_input":"대한민국의 수도는?"
})

In [8]:
# 텍스트가 아닌 AIMessage 객체로 출력
response

AIMessage(content='대한민국의 수도는 **서울(Seoul)**입니다! 😊\n\n궁금한 점이 또 있으시면 언제든지 물어봐 주세요. 제가 친절하게 도와드릴게요! ✨', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-04-05T15:31:44.7175837Z', 'done': True, 'done_reason': 'stop', 'total_duration': 52991694800, 'load_duration': 13684691800, 'prompt_eval_count': 43, 'prompt_eval_duration': 2620794300, 'eval_count': 308, 'eval_duration': 35815015100, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--d1c68aea-810a-433e-a0b8-5efb05febf4f-0', usage_metadata={'input_tokens': 43, 'output_tokens': 308, 'total_tokens': 351})

In [9]:
# 모델에서 받은 대답에 대한 내용만 출력
print(response.content)

대한민국의 수도는 **서울(Seoul)**입니다! 😊

궁금한 점이 또 있으시면 언제든지 물어봐 주세요. 제가 친절하게 도와드릴게요! ✨


## Chain with Parser
> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 출력 파싱 -> 결과 반환

In [10]:
from langchain_core.output_parsers import StrOutputParser

In [13]:
# chain형성
# 형성한 프롬프트 | 모델 | 출력 파싱
chain = chat_prompt | model | StrOutputParser()

In [ ]:
response = chain.invoke({
    "user_input":"대한민국의 수도는?"
})

In [15]:
# OutputParser를 사용하여 텍스트로 출력
print(response)

대한민국의 수도는 **서울(Seoul)**입니다! 🇰🇷✨

궁금한 점이 또 있으시면 언제든지 물어봐 주세요. 제가 친절하게 도와드릴게요! 😊


In [16]:
type(response)

str